In [ ]:
# Assignment 1: Web scraping and data wrangling
!pip install nltk requests beautifulsoup4

import requests
from bs4 import BeautifulSoup
import pandas as pd
import json

page_nos = []
for i in range(25):
    page_nos.append(i+1)
    

data = {}

for page_no in page_nos:
    rel_path = f"/catalogue/page-{page_no}.html"
    url = "https://books.toscrape.com" + rel_path
    
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.find_all("article", class_ = "product_pod")
    
    i = 1
    for book in books:
        
        url1 = book.find("a")["href"]
        response1 = requests.get("https://books.toscrape.com/catalogue/" + url1)
        soup1 = BeautifulSoup(response1.text,"html.parser")
        page = soup1.find("article")
        cat = soup1.find("ul",class_="breadcrumb").find_all("a")[2].text
        upc = page.find("table",class_="table table-striped").find("tr").find("td").text
        n_avail = page.find("table", class_ = "table table-striped").find_all("tr")[5].find('td').text[9:]
        
        item = {
            "Title": book.h3.a["title"],
            "Price": book.find("p", class_="price_color").text,
            "Rating": book.find("p")["class"][1],
            "Availability": book.find("p", class_ = "instock availability").text.strip(),
            "Product URL": "https://books.toscrape.com" + book.h3.a["href"],
            "UPC": upc,
            "Category": cat,
            "Available Books": n_avail 
            
        }
        data[f"{page_no}.{i}"] = item
        i = i + 1
        
    
print(data)   

with open("books.json","w", encoding="utf-8") as f:
    json.dump(data,f,indent=4, ensure_ascii=False)

import pandas as pd

df = pd.DataFrame(data)

df = df.T
df = df.reset_index(drop=True)
print(df.head())




rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
avail_map = {"In stock": True}

df['Price'] = df['Price'].astype(str).str[2:].astype(float)
df['Rating'] = df["Rating"].map(rating_map)
df["Availability"] = df["Availability"].map(avail_map)
df["Available Books"] = df["Available Books"].astype(str).str.extract(r'(\d+)').astype(int)

df.head()    

df['Title Length'] = df['Title'].str.len()

df['Price per Rating'] = (df['Price'] / df['Rating']).round(2)


q1 = df["Price"].quantile(0.33)
q2 = df["Price"].quantile(0.67)

bins = [-float('inf'), q1, q2, float('inf')]

labels = ["low", "med", "high"]

df["Price Category"] = pd.cut(df["Price"], bins= bins, labels = labels)

df[['Title', 'Title Length','Price per Rating', 'Rating', "Price","Price Category"]]

min_val = df['Title Length'].min()
max_val = df['Title Length'].max()

df['Title Length Scaled'] = (df['Title Length'] - min_val) / (max_val - min_val)
df['Title Length Scaled'] = df['Title Length Scaled'].round(3)

df.head()

min_price = df['Price'].min()
max_price = df['Price'].max()

df['Price Scaled'] = (df['Price'] - min_price) / (max_price - min_price)

df['Price Scaled'] = df['Price Scaled'].round(3)

print(df.head())

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.hist(df['Price'], bins=20, color='red', edgecolor='black')
plt.title('Distribution of Book Prices')
plt.xlabel('Price ($)')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

rating_counts = df['Rating'].value_counts().sort_index()

plt.figure(figsize=(8, 5))
plt.bar(rating_counts.index, rating_counts.values, color='gold', edgecolor='black')
plt.title('Distribution of Book Ratings')
plt.xlabel('Rating')
plt.ylabel('Number of Books')
plt.xticks(rating_counts.index)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

top_categories = df['Category'].value_counts()#.head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_categories.index[::-1], top_categories.values[::-1], color='green', edgecolor='black')
plt.title('Top 10 Book Categories by Volume')
plt.xlabel('Number of Books')
plt.ylabel('Category')
plt.yticks(fontsize=8)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

num_cols = df[['Price', 'Rating', 'Title Length', 'Price per Rating']]
corr = num_cols.corr()

plt.figure(figsize=(7, 6))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45)
plt.yticks(range(len(corr.columns)), corr.columns)

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha='center', va='center', color='black')

plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

price_cat_counts = df['Price Category'].value_counts()

plt.figure(figsize=(7, 5))
plt.pie(price_cat_counts, labels=price_cat_counts.index, autopct='%1.1f%%')
plt.title('Proportion of Books by Price Category')
plt.show()

df.to_csv('cleaned_books_dataset.csv', index=False)



In [ ]:
# Assignment 2: California Housing regression
# CELL 1: Import libraries and load dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load California Housing dataset
housing = fetch_california_housing(as_frame=True)

df = housing.frame

# Display first few records
print("First 5 records:")
print(df.head())

# Dataset dimensions
print("\nDataset Dimensions:", df.shape)

# Feature names
print("\nFeatures:")
print(df.columns.tolist())

# Data types
print("\nData Types:")
print(df.dtypes)

# Summary statistics
print("\nSummary Statistics:")
print(df.describe())

# CELL 2: Missing values, duplicates and EDA visualizations

# Missing values
print("Missing Values:")
print(df.isnull().sum())

# Total missing values
print("\nTotal Missing Values:", df.isnull().sum().sum())

# Duplicate records
print("\nNumber of Duplicate Records:", df.duplicated().sum())

# ---------------- HISTOGRAMS ----------------

df.hist(figsize=(15, 12), bins=30, edgecolor="black")
plt.suptitle("Distribution of California Housing Features", fontsize=16)
plt.tight_layout()
plt.show()

# ---------------- BOX PLOTS ----------------

plt.figure(figsize=(15, 8))
sns.boxplot(data=df)
plt.xticks(rotation=45)
plt.title("Box Plot of Housing Features")
plt.tight_layout()
plt.show()

# ---------------- SCATTER PLOTS ----------------

# Median Income vs Median House Value
plt.figure(figsize=(8, 6))
plt.scatter(df["MedInc"], df["MedHouseVal"], alpha=0.3)
plt.xlabel("Median Income")
plt.ylabel("Median House Value")
plt.title("Median Income vs Median House Value")
plt.show()

# Latitude vs Longitude
plt.figure(figsize=(8, 6))
plt.scatter(df["Longitude"], df["Latitude"],
            c=df["MedHouseVal"], alpha=0.4)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Geographical Distribution of House Values")
plt.colorbar(label="Median House Value")
plt.show()

# CELL 3: Correlation analysis and outlier detection

# Correlation matrix
correlation = df.corr()

print("Correlation Matrix:")
print(correlation)

# Correlation heatmap
plt.figure(figsize=(12, 9))
sns.heatmap(correlation,
            annot=True,
            cmap="coolwarm",
            fmt=".2f",
            linewidths=0.5)

plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

# Correlation with target
print("\nCorrelation with Median House Value:")
print(correlation["MedHouseVal"].sort_values(ascending=False))

# ---------------- OUTLIER DETECTION ----------------

Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_count = ((df < lower_bound) | (df > upper_bound)).sum()

print("\nOutlier Count for Each Feature:")
print(outlier_count)

print("\nTotal Outlier Values:", outlier_count.sum())

# ---------------- OUTLIER TREATMENT ----------------

# Create a copy
df_clean = df.copy()

# Winsorization using IQR boundaries
for column in df_clean.columns:
    df_clean[column] = df_clean[column].clip(
        lower=lower_bound[column],
        upper=upper_bound[column]
    )

print("\nOutliers treated using IQR clipping.")

# CELL 4: Preprocessing and train-test split

# ---------------- HANDLE MISSING VALUES ----------------

# Replace missing numerical values with median
for column in df_clean.columns:
    df_clean[column] = df_clean[column].fillna(df_clean[column].median())

print("Missing values after treatment:")
print(df_clean.isnull().sum())

# ---------------- SEPARATE FEATURES AND TARGET ----------------

X = df_clean.drop("MedHouseVal", axis=1)
y = df_clean["MedHouseVal"]

# ---------------- TRAIN-TEST SPLIT ----------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

# ---------------- FEATURE SCALING ----------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled data back to DataFrames
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

print("\nScaled Training Data:")
print(X_train_scaled.head())

print("\nScaled Testing Data:")
print(X_test_scaled.head())

X_train_scaled.to_csv("X_train_scaled.csv", index=False)
X_test_scaled.to_csv("X_test_scaled.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Preprocessed dataset saved to CSV.")





import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load preprocessed data
X_train = pd.read_csv("X_train_scaled.csv")
X_test = pd.read_csv("X_test_scaled.csv")

y_train = pd.read_csv("y_train.csv").squeeze()
y_test = pd.read_csv("y_test.csv").squeeze()

# Combine training data temporarily to calculate correlations
train_data = X_train.copy()
train_data["MedHouseVal"] = y_train.values

# Correlation of each feature with target
correlations = train_data.corr()["MedHouseVal"].drop("MedHouseVal")

# Sort by absolute correlation
correlations_abs = correlations.abs().sort_values(ascending=False)

print("Correlation with Median House Value:")
print(correlations.sort_values(ascending=False))

# Model A: 3 most highly correlated features
model_A_features = correlations_abs.head(3).index.tolist()

# Model B: 5 most highly correlated features
model_B_features = correlations_abs.head(5).index.tolist()

# Model C: All features
model_C_features = X_train.columns.tolist()

print("\nModel A Features:", model_A_features)
print("Model B Features:", model_B_features)
print("Model C Features:", model_C_features)

# Create models
model_A = LinearRegression()
model_B = LinearRegression()
model_C = LinearRegression()

# ---------------- MODEL A ----------------
start_A = time.perf_counter()

model_A.fit(
    X_train[model_A_features],
    y_train
)

time_A = time.perf_counter() - start_A

# ---------------- MODEL B ----------------
start_B = time.perf_counter()

model_B.fit(
    X_train[model_B_features],
    y_train
)

time_B = time.perf_counter() - start_B

# ---------------- MODEL C ----------------
start_C = time.perf_counter()

model_C.fit(
    X_train[model_C_features],
    y_train
)

time_C = time.perf_counter() - start_C

print("All three models trained successfully.")

# Predictions
y_pred_A = model_A.predict(X_test[model_A_features])
y_pred_B = model_B.predict(X_test[model_B_features])
y_pred_C = model_C.predict(X_test[model_C_features])


def evaluate_model(model_name, y_true, y_pred, execution_time, features):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    return {
        "Model": model_name,
        "Number of Features": len(features),
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2,
        "Training Time (seconds)": execution_time
    }


results = []

results.append(
    evaluate_model(
        "Model A",
        y_test,
        y_pred_A,
        time_A,
        model_A_features
    )
)

results.append(
    evaluate_model(
        "Model B",
        y_test,
        y_pred_B,
        time_B,
        model_B_features
    )
)

results.append(
    evaluate_model(
        "Model C",
        y_test,
        y_pred_C,
        time_C,
        model_C_features
    )
)

results_df = pd.DataFrame(results)

print("\nComparative Performance Table:")
print(results_df.round(6).to_string(index=False))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.barplot(x="Model", y="RMSE", data=results_df, palette="Reds_d")
plt.title("RMSE Comparison(Lower is Better)")
plt.xlabel("Model")
plt.ylabel("RMSE")

plt.subplot(1, 2, 2)
sns.barplot(x="Model", y="R2 Score", data=results_df, palette="Blues_d")
plt.title("R² Score Comparison(Higher is Better)")
plt.xlabel("Model")
plt.ylabel("R² Score")

plt.tight_layout()
plt.savefig("model_performance_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

# Find best model based on R2 score
best_model = results_df.loc[
    results_df["R2 Score"].idxmax()
]

print("FEATURE COMBINATIONS")
print("-" * 50)

print("Model A:", model_A_features)
print("Model B:", model_B_features)
print("Model C:", model_C_features)

print("\nBEST MODEL")
print("-" * 50)
print("Model:", best_model["Model"])
print("Number of Features:", int(best_model["Number of Features"]))
print("MAE:", round(best_model["MAE"], 6))
print("MSE:", round(best_model["MSE"], 6))
print("RMSE:", round(best_model["RMSE"], 6))
print("R2 Score:", round(best_model["R2 Score"], 6))
print("Training Time:", round(best_model["Training Time (seconds)"], 6), "seconds")

import joblib

# Save Model C
joblib.dump(model_C, "california_housing_model_C.pkl")

print("Model C saved successfully as california_housing_model_C.pkl")





import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load preprocessed data
X_train = pd.read_csv("X_train_scaled.csv")
X_test = pd.read_csv("X_test_scaled.csv")
y_train = pd.read_csv("y_train.csv").squeeze()
y_test = pd.read_csv("y_test.csv").squeeze()

# Load saved Model C
model_C = joblib.load("california_housing_model_C.pkl")

# Model C uses all features
all_features = X_train.columns.tolist()

# RFE for feature selection
base_model = LinearRegression()

rfe = RFE(
    estimator=base_model,
    n_features_to_select=5
)

rfe.fit(X_train, y_train)

# Selected features
selected_features = X_train.columns[rfe.support_].tolist()

print("Selected Features using RFE:")
print(selected_features)



In [ ]:
# Assignment 3: Classification, PCA/SVD, and regularized regression
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_excel('Telco_customer_churn.xlsx')

df = df.drop_duplicates()

drop_cols = ['CustomerID', 'Count', 'Country', 'State', 'Lat Long', 'Churn Label', 'Churn Score', 'CLTV', 'Churn Reason']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Total Charges'] = df['Total Charges'].fillna(df['Total Charges'].median())

target_col = 'Churn Value'
X = df.drop(columns=[target_col])
y = df[target_col]

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print("Preprocessing complete. Training shape:", X_train_scaled.shape, "Test shape:", X_test_scaled.shape)

import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

sns.countplot(x='Churn Value', data=df, ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Target Variable Distribution (Churn Value)')

sns.countplot(x='Contract', hue='Churn Value', data=df, ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('Churn Rate by Contract Type')

sns.kdeplot(data=df, x='Tenure Months', hue='Churn Value', common_norm=False, ax=axes[1, 0], fill=True)
axes[1, 0].set_title('Tenure Distribution by Churn')

sns.boxplot(x='Churn Value', y='Monthly Charges', data=df, ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('Monthly Charges by Churn Status')

plt.tight_layout()
plt.show()

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

results = []
model_probs = {}
model_preds = {}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, (name, model) in enumerate(models.items()):
    model.fit(X_train_scaled, y_train)
    
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    model_preds[name] = y_pred
    model_probs[name] = y_prob
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc
    })
    
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=axes[idx], cmap='Blues', colorbar=False)
    axes[idx].set_title(f'CM: {name}')

axes[-1].axis('off')
plt.tight_layout()
plt.show()

metrics_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False).reset_index(drop=True)
print(metrics_df.to_string(index=False))

from sklearn.metrics import roc_curve, precision_recall_curve

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for name, probs in model_probs.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax1.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_score(y_test, probs):.3f})')
    
    precision, recall, _ = precision_recall_curve(y_test, probs)
    ax2.plot(recall, precision, label=f'{name}')

ax1.plot([0, 1], [0, 1], 'k--', label='Chance')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves')
ax1.legend(loc='lower right')
ax1.grid(True)

ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves')
ax2.legend(loc='lower left')
ax2.grid(True)

plt.tight_layout()
plt.show()

best_model_name = metrics_df.iloc[0]['Model']
best_auc = metrics_df.iloc[0]['ROC-AUC']
print(f"Best Performing Model: {best_model_name} with ROC-AUC = {best_auc:.4f}")



import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_excel('Telco_customer_churn.xlsx')
df = df.drop_duplicates()

drop_cols = ['CustomerID', 'Count', 'Country', 'State', 'Lat Long', 'Churn Label', 'Churn Score', 'CLTV', 'Churn Reason']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Total Charges'] = df['Total Charges'].fillna(df['Total Charges'].median())

target_col = 'Churn Value'
X = df.drop(columns=[target_col])
y = df[target_col]

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print("Full features shape:", X_train_scaled.shape)

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, TruncatedSVD

pca_full = PCA().fit(X_train_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
n_components = np.argmax(cum_var >= 0.90) + 1

pca = PCA(n_components=n_components, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

svd = TruncatedSVD(n_components=n_components, random_state=42)
X_train_svd = svd.fit_transform(X_train_scaled)
X_test_svd = svd.transform(X_test_scaled)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cum_var) + 1), cum_var, marker='o', linestyle='--')
plt.axhline(y=0.90, color='r', linestyle=':', label='90% Variance Threshold')
plt.axvline(x=n_components, color='g', linestyle=':', label=f'{n_components} Components Selected')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance')
plt.legend()
plt.grid(True)
plt.show()

print(f"Selected {n_components} components (retaining ~90% variance).")

import time
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

import seaborn as sns
import matplotlib.pyplot as plt

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

feature_sets = {
    'Original': (X_train_scaled, X_test_scaled, X_train_scaled.shape[1]),
    'PCA': (X_train_pca, X_test_pca, n_components),
    'SVD': (X_train_svd, X_test_svd, n_components)
}

all_results = []
roc_curves = {}

# 1. Fit models, store metrics, and generate 3x5 Confusion Matrix grid
fig_cm, axes_cm = plt.subplots(3, 5, figsize=(18, 10))

for r_idx, (set_name, (X_tr, X_te, n_feats)) in enumerate(feature_sets.items()):
    for c_idx, (m_name, model) in enumerate(models.items()):
        start_time = time.time()
        model.fit(X_tr, y_train)
        fit_time = time.time() - start_time
        
        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else y_pred
        
        roc_curves[(set_name, m_name)] = (y_test, y_prob)
        
        all_results.append({
            'Feature Set': set_name,
            'Model': m_name,
            'Features/Components': n_feats,
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred),
            'Recall': recall_score(y_test, y_pred),
            'F1-Score': f1_score(y_test, y_pred),
            'ROC-AUC': roc_auc_score(y_test, y_prob),
            'Training Time (s)': fit_time
        })
        
        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot(ax=axes_cm[r_idx, c_idx], cmap='Blues', colorbar=False)
        axes_cm[r_idx, c_idx].set_title(f'{set_name}\n{m_name}', fontsize=9)

plt.tight_layout()
plt.show()

results_df = pd.DataFrame(all_results)

# 2. Multi-Metric Comparison Plots
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Training Time (s)']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()
palette = sns.color_palette("Set2", 3)

for idx, metric in enumerate(metrics_to_plot):
    sns.barplot(data=results_df, x="Model", y=metric, hue="Feature Set", ax=axes[idx], palette=palette)
    axes[idx].set_title(f'{metric} Comparison by Model', fontsize=11, fontweight='bold')
    axes[idx].grid(axis='y', linestyle='--', alpha=0.7)
    axes[idx].tick_params(axis='x', rotation=15)
    
    if metric != 'Training Time (s)':
        axes[idx].set_ylim(0.40, 0.90)
    
    if idx != 0:
        axes[idx].legend().remove()
    else:
        axes[idx].legend(title="Feature Set", loc='lower right')

plt.tight_layout()
plt.show()

from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for set_name in ['Original', 'PCA', 'SVD']:
    m_name = 'Logistic Regression'
    y_true, y_prob = roc_curves[(set_name, m_name)]
    
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    axes[0].plot(fpr, tpr, label=f'{set_name} - {m_name}')
    
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    axes[1].plot(rec, prec, label=f'{set_name} - {m_name}')

axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_title('ROC Curves (Logistic Regression Across Feature Sets)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()
axes[0].grid(True)

axes[1].set_title('Precision-Recall Curves (Logistic Regression)')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Cell 1: Import required libraries, configure plot settings, and load the dataset
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Enhance visualization aesthetics
plt.rcParams["figure.dpi"] = 300
sns.set_theme(style="whitegrid", palette="muted")

# Load the dataset
df = pd.read_csv("Housing.csv")
print("Dataset Head:")
print(df.head())

# Cell 2: Preprocess dataset (missing values, encoding binary/categorical features, train-test split, scaling)

# 1. Handle Missing Values (if any)
df = df.dropna()

# 2. Identify Binary and Categorical Features
binary_cols = [
    "mainroad",
    "guestroom",
    "basement",
    "hotwaterheating",
    "airconditioning",
    "prefarea",
]
for col in binary_cols:
    df[col] = df[col].map({"yes": 1, "no": 0})

# Encode nominal categorical variable 'furnishingstatus'
df = pd.get_dummies(df, columns=["furnishingstatus"], drop_first=True)

# 3. Separate Features and Target Variable
X = df.drop("price", axis=1)
y = df["price"]

# 4. Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Standard Scaling (Fit on Train, Transform on Test)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Data Preprocessing Completed successfully.")
print(f"Features shape: {X_train_scaled.shape}")

# Cell 3: Train Multiple Linear Regression model and record evaluation metrics


def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2


# Train Multiple Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# Predict and Evaluate
y_pred_lr = lr.predict(X_test_scaled)
mse_lr, rmse_lr, mae_lr, r2_lr = compute_metrics(y_test, y_pred_lr)

print("--- Multiple Linear Regression Performance ---")
print(f"MSE  : {mse_lr:,.2f}")
print(f"RMSE : {rmse_lr:,.2f}")
print(f"MAE  : {mae_lr:,.2f}")
print(f"R²   : {r2_lr:.4f}")

# Cell 4: Train Ridge, Lasso, and Elastic Net across multiple lambda (alpha) values and record errors

lambdas = np.logspace(-3, 3, 50)  # Range of alpha/lambda values

results = {"Ridge": {}, "Lasso": {}, "ElasticNet": {}}

for lmbda in lambdas:
    # 1. Ridge
    ridge = Ridge(alpha=lmbda, random_state=42)
    ridge.fit(X_train_scaled, y_train)
    r_tr_pred = ridge.predict(X_train_scaled)
    r_te_pred = ridge.predict(X_test_scaled)

    # 2. Lasso
    lasso = Lasso(alpha=lmbda, random_state=42, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    l_tr_pred = lasso.predict(X_train_scaled)
    l_te_pred = lasso.predict(X_test_scaled)

    # 3. Elastic Net
    enet = ElasticNet(
        alpha=lmbda, l1_ratio=0.5, random_state=42, max_iter=10000
    )
    enet.fit(X_train_scaled, y_train)
    e_tr_pred = enet.predict(X_train_scaled)
    e_te_pred = enet.predict(X_test_scaled)

    # Store Errors (RMSE)
    results["Ridge"][lmbda] = (
        np.sqrt(mean_squared_error(y_train, r_tr_pred)),
        np.sqrt(mean_squared_error(y_test, r_te_pred)),
    )
    results["Lasso"][lmbda] = (
        np.sqrt(mean_squared_error(y_train, l_tr_pred)),
        np.sqrt(mean_squared_error(y_test, l_te_pred)),
    )
    results["ElasticNet"][lmbda] = (
        np.sqrt(mean_squared_error(y_train, e_tr_pred)),
        np.sqrt(mean_squared_error(y_test, e_te_pred)),
    )

print("Regularization grid search completed.")

# Cell 5: Visualizing Training vs Testing RMSE across different values of lambda (alpha)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models = ["Ridge", "Lasso", "ElasticNet"]

for idx, model_name in enumerate(models):
    train_errors = [results[model_name][l][0] for l in lambdas]
    test_errors = [results[model_name][l][1] for l in lambdas]

    axes[idx].plot(
        lambdas,
        train_errors,
        label="Train RMSE",
        color="#1f77b4",
        linewidth=2,
    )
    axes[idx].plot(
        lambdas,
        test_errors,
        label="Test RMSE",
        color="#ff7f0e",
        linewidth=2,
        linestyle="--",
    )
    axes[idx].set_xscale("log")
    axes[idx].set_xlabel("Lambda (Alpha)", fontsize=11)
    axes[idx].set_ylabel("RMSE", fontsize=11)
    axes[idx].set_title(f"{model_name} Regression Error", fontsize=13)
    axes[idx].legend()

plt.tight_layout()
plt.show()

# Cell 6: Train best models, compare coefficients, display evaluation table, and summarize regularization impact

# Best lambda choice for optimal penalty evaluation
opt_lambda = 1.0

# Refit regularized models with optimal lambda
best_ridge = Ridge(alpha=opt_lambda, random_state=42).fit(
    X_train_scaled, y_train
)
best_lasso = Lasso(alpha=opt_lambda, random_state=42, max_iter=10000).fit(
    X_train_scaled, y_train
)
best_enet = ElasticNet(
    alpha=opt_lambda, l1_ratio=0.5, random_state=42, max_iter=10000
).fit(X_train_scaled, y_train)

# 1. Performance Comparison Table
summary_data = []
all_models = {
    "Linear Regression": lr,
    "Ridge (λ=1.0)": best_ridge,
    "Lasso (λ=1.0)": best_lasso,
    "Elastic Net (λ=1.0)": best_enet,
}

for name, model in all_models.items():
    preds = model.predict(X_test_scaled)
    mse, rmse, mae, r2 = compute_metrics(y_test, preds)
    summary_data.append(
        {
            "Model": name,
            "MSE": f"{mse:,.2f}",
            "RMSE": f"{rmse:,.2f}",
            "MAE": f"{mae:,.2f}",
            "R² Score": f"{r2:.4f}",
        }
    )

comparison_df = pd.DataFrame(summary_data)
print("--- Final Model Comparison Table ---")
print(comparison_df.to_string(index=False))

# 2. Coefficient Visualizations
coef_df = pd.DataFrame(
    {
        "Feature": X.columns,
        "Linear": lr.coef_,
        "Ridge": best_ridge.coef_,
        "Lasso": best_lasso.coef_,
        "ElasticNet": best_enet.coef_,
    }
).melt(id_vars="Feature", var_name="Model", value_name="Coefficient Value")

plt.figure(figsize=(14, 6))
sns.barplot(
    data=coef_df, x="Feature", y="Coefficient Value", hue="Model", palette="tab10"
)
plt.title("Comparison of Regression Coefficients Across Models", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.axhline(0, color="black", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.show()



In [ ]:
# Assignment 4: t-tests
import numpy as np
from scipy import stats

# Data
group_A = [65, 70, 68, 72, 66, 75, 69, 71, 67, 70]
group_B = [72, 75, 78, 74, 80, 77, 73, 79, 76, 81]
alpha = 0.05

# 1. Means
mean_A = sum(group_A) / len(group_A)
mean_B = sum(group_B) / len(group_B)

# 2. Sample Variances (ddof=1 for sample variance)
n_A, n_B = len(group_A), len(group_B)
var_A = sum((x - mean_A) ** 2 for x in group_A) / (n_A - 1)
var_B = sum((x - mean_B) ** 2 for x in group_B) / (n_B - 1)

# 3. Pooled Standard Deviation
df = n_A + n_B - 2
s_p = np.sqrt(((n_A - 1) * var_A + (n_B - 1) * var_B) / df)

# 4. T-Statistic
t_stat = (mean_A - mean_B) / (s_p * np.sqrt(1/n_A + 1/n_B))

# 5. P-Value (two-tailed)
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=df))

print(f"Mean Group A: {mean_A:.2f}")
print(f"Mean Group B: {mean_B:.2f}")
print(f"Calculated t-statistic: {t_stat:.4f}")
print(f"Degrees of freedom: {df}")
print(f"p-value: {p_value:.6f}")

if p_value < alpha:
    print("Conclusion: Reject the null hypothesis. There is a statistically significant difference between the two teaching methods.")
else:
    print("Conclusion: Fail to reject the null hypothesis. There is no statistically significant difference.")



from scipy import stats

# Data
before = [52, 48, 65, 60, 55, 70, 58, 62, 50, 67]
after  = [60, 55, 70, 66, 63, 75, 64, 68, 57, 72]
alpha = 0.05

# 1. Differences
differences = [a - b for a, b in zip(after, before)]
n = len(differences)

# 2. Mean Difference
mean_d = sum(differences) / n

# 3. Standard Deviation of Differences
var_d = sum((d - mean_d) ** 2 for d in differences) / (n - 1)
sd_d = var_d ** 0.5

# 4. Standard Error of Mean Difference
se_d = sd_d / (n ** 0.5)

# 5. T-Statistic
t_stat = mean_d / se_d

# 6. P-Value (two-tailed)
df = n - 1
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=df))

print(f"Mean Difference: {mean_d:.2f}")
print(f"Calculated t-statistic: {t_stat:.4f}")
print(f"Degrees of freedom: {df}")
print(f"p-value: {p_value:.6f}")

if p_value < alpha:
    print("Conclusion: Reject the null hypothesis. The training program resulted in a statistically significant change in programming scores.")
else:
    print("Conclusion: Fail to reject the null hypothesis. No statistically significant change observed.")



import numpy as np
from scipy import stats

score_before = [65, 72, 58, 80, 62, 70, 75, 68, 59, 74, 66, 61]
score_after  = [69, 73, 62, 81, 64, 75, 76, 71, 62, 75, 67, 65]
alpha = 0.05
n = len(score_before)

# --- 1. MANUAL INDEPENDENT TWO-SAMPLE T-TEST ---
mean_b = sum(score_before) / n
mean_a = sum(score_after) / n

var_b = sum((x - mean_b)**2 for x in score_before) / (n - 1)
var_a = sum((x - mean_a)**2 for x in score_after) / (n - 1)

df_ind = 2 * n - 2
pooled_var = ((n - 1) * var_b + (n - 1) * var_a) / df_ind
se_ind = np.sqrt(pooled_var * (2 / n))

t_stat_ind = (mean_a - mean_b) / se_ind
p_val_ind = 2 * (1 - stats.t.cdf(abs(t_stat_ind), df=df_ind))

# --- 2. MANUAL PAIRED T-TEST ---
differences = [a - b for a, b in zip(score_after, score_before)]
mean_diff = sum(differences) / n

var_diff = sum((d - mean_diff)**2 for d in differences) / (n - 1)
se_diff = np.sqrt(var_diff / n)

df_paired = n - 1
t_stat_paired = mean_diff / se_diff
p_val_paired = 2 * (1 - stats.t.cdf(abs(t_stat_paired), df=df_paired))

# --- RESULTS DISPLAY ---
print("================ INDEPENDENT T-TEST ================")
print(f"Mean Difference (After - Before): {mean_a - mean_b:.2f}")
print(f"Degrees of Freedom:              {df_ind}")
print(f"t-statistic:                      {t_stat_ind:.4f}")
print(f"p-value:                          {p_val_ind:.6f}")

print("\n=================== PAIRED T-TEST ===================")
print(f"Mean Difference:                  {mean_diff:.2f}")
print(f"Degrees of Freedom:              {df_paired}")
print(f"t-statistic:                      {t_stat_paired:.4f}")
print(f"p-value:                          {p_val_paired:.6f}")

